In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
!pip install numpy pandas scipy matplotlib seaborn scikit-learn xgboost neurokit2 tensorflow

After running the above cell, you will be prompted to authenticate and allow Colab to access your Google Drive. Once mounted, you can specify the path to your data file. For example, if you have a CSV file named `my_data.csv` in the root of your Google Drive, you can load it into a pandas DataFrame like this:

In [36]:
import pandas as pd

data_path = '/content/drive/MyDrive/major/cleaned_output.csv'

try:
    df = pd.read_csv(data_path)
    print(f"Successfully loaded data from {data_path}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file at '{data_path}' was not found. Please check the path and try again.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

Successfully loaded data from /content/drive/MyDrive/major/cleaned_output.csv


,timestamp,sensor1,sensor2,status
0,25702023,1922.0,2096.0,STILL
1,25705897,1922.0,2096.0,STILL
2,25706881,1907.0,2096.0,STILL
3,25710023,1901.0,2096.0,STILL
4,25714024,1888.0,2096.0,STILL


### Initial Data Overview
Let's get a quick overview of the DataFrame, including data types, non-null values, and memory usage. This will help identify any columns that might need type conversion or have missing values.

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200046 entries, 0 to 200045
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   timestamp  200046 non-null  int64  
 1   sensor1    200046 non-null  float64
 2   sensor2    200046 non-null  float64
 3   status     200046 non-null  object 
dtypes: float64(2), int64(1), object(1)
memory usage: 6.1+ MB


Next, let's look at the descriptive statistics of the numerical columns. This will give us insights into the distribution and range of values in `sensor1`, `sensor2`, and `timestamp`.

In [38]:
display(df.describe())

,timestamp,sensor1,sensor2
count,2.000460e+05,200046.000000,200046.000000
mean,3.442442e+08,2080.155204,2038.546659
std,1.839123e+08,330.318099,113.772116
min,2.570202e+07,0.000000,1596.000000
25%,1.849721e+08,1971.000000,1969.000000
50%,3.442440e+08,2023.000000,2063.000000
75%,5.035170e+08,2087.000000,2111.000000
max,6.627860e+08,4095.000000,2317.000000


Now that we have an initial overview, please share the code you used for making your model or describe the preprocessing steps you'd like to apply. This will help me preprocess the current dataset in a similar way.

In [48]:
import pandas as pd
import numpy as np
import neurokit2 as nk
from scipy.integrate import trapezoid
import warnings

# Suppress specific NeuroKit2 complexity warnings
warnings.filterwarnings('ignore', category=RuntimeWarning, module='neurokit2')

# ================= CONFIG =================
DATA_PATH = "/content/drive/MyDrive/major/cleaned_output.csv"
FS = 32
WINDOW_SEC = 120
STEP_SEC = 60

# ================= LOAD DATA =================
def load_subject(path):
    df = pd.read_csv(path)
    df = df.rename(columns={"sensor1": "hrv", "sensor2": "gsr"})
    return df

# ================= OPTIMIZED EDA PROCESSING =================
def build_eda_table(df):
    eda_signal = df["gsr"].values
    try:
        signals, info = nk.eda_process(eda_signal, sampling_rate=FS)
    except Exception as e:
        print(f"EDA global processing error: {e}")
        return pd.DataFrame()

    WINDOW = FS * WINDOW_SEC
    STEP = FS * STEP_SEC
    rows = []

    for i in range(0, len(eda_signal) - WINDOW, STEP):
        win_phasic = signals["EDA_Phasic"].iloc[i : i + WINDOW].values
        win_tonic = signals["EDA_Tonic"].iloc[i : i + WINDOW].values
        scr_peaks = signals["SCR_Peaks"].iloc[i : i + WINDOW].sum()

        rows.append({
            "Time": i / FS,
            "Label": 0,
            "EDA_SCR_count": scr_peaks,
            "EDA_Phasic_AUC": trapezoid(np.abs(win_phasic)) / FS,
            "EDA_Tonic_Mean": np.mean(win_tonic)
        })
    return pd.DataFrame(rows)

# ================= OPTIMIZED HRV PROCESSING =================
def build_hrv_table(df):
    ecg_signal = df["hrv"].values
    cleaned = nk.ecg_clean(ecg_signal, sampling_rate=FS)

    try:
        _, info = nk.ecg_peaks(cleaned, sampling_rate=FS)
        rpeaks = info["ECG_R_Peaks"]
    except Exception as e:
        print(f"Global R-peak detection failed: {e}")
        return pd.DataFrame()

    WINDOW = FS * WINDOW_SEC
    STEP = FS * STEP_SEC
    rows = []

    for start in range(0, len(ecg_signal) - WINDOW, STEP):
        end = start + WINDOW
        win_peaks = rpeaks[(rpeaks >= start) & (rpeaks < end)]

        if len(win_peaks) < 5:
            continue

        try:
            # Use standard HRV features to avoid complexity calculation errors
            hrv_results = nk.hrv(win_peaks, sampling_rate=FS)
            rows.append({
                "Time": (start + end) / 2 / FS,
                "Label": 0,
                "HRV_RMSSD": hrv_results["HRV_RMSSD"].values[0],
                "HRV_SDNN": hrv_results["HRV_SDNN"].values[0],
                "HRV_MeanNN": hrv_results["HRV_MeanNN"].values[0],
                "HRV_LF": hrv_results.get("HRV_LF", pd.Series([0])).values[0],
                "HRV_HF": hrv_results.get("HRV_HF", pd.Series([0])).values[0],
                "HRV_LFHF": hrv_results.get("HRV_LFHF", pd.Series([0])).values[0],
            })
        except:
            continue
    return pd.DataFrame(rows)

# ================= PIPELINE =================
df = load_subject(DATA_PATH)
eda_df = build_eda_table(df)
hrv_df = build_hrv_table(df)

if not eda_df.empty and not hrv_df.empty:
    eda_df["Subject"], hrv_df["Subject"] = "S1", "S1"
    merged_df = pd.merge_asof(
        eda_df.sort_values("Time"),
        hrv_df.sort_values("Time"),
        on="Time",
        by=["Label", "Subject"],
        direction="nearest"
    )
    merged_df['EDA_Tonic_Diff'] = merged_df['EDA_Tonic_Mean'].diff().fillna(0)
    merged_df['HRV_RMSSD_Diff'] = merged_df['HRV_RMSSD'].diff().fillna(0)

    print(f"\n===== OPTIMIZED DATASET SUMMARY ====")
    print("Total samples extracted:", len(merged_df))
    display(merged_df.head())
else:
    print("❌ Refactor failed: Insufficient data for merge.")


===== OPTIMIZED DATASET SUMMARY ====
Total samples extracted: 103


,Time,Label,EDA_SCR_count,EDA_Phasic_AUC,EDA_Tonic_Mean,Subject,HRV_RMSSD,HRV_SDNN,HRV_MeanNN,HRV_LF,HRV_HF,HRV_LFHF,EDA_Tonic_Diff,HRV_RMSSD_Diff
0,0.0,0,35,295.819091,2082.367377,S1,6705.786913,4324.007142,13494.791667,0.001246,1.306774e-06,953.305097,0.000000,0.000000
1,60.0,0,33,301.882044,2101.820558,S1,6705.786913,4324.007142,13494.791667,0.001246,1.306774e-06,953.305097,19.453181,0.000000
2,120.0,0,21,264.412910,2122.767895,S1,12097.907877,11144.113964,18927.083333,0.000707,9.733532e-08,7262.442184,20.947336,5392.120964
3,180.0,0,20,279.423785,2144.672340,S1,14850.524770,11202.834434,24773.437500,0.000867,2.821577e-06,307.423673,21.904445,2752.616893
4,240.0,0,30,340.786547,2182.895880,S1,14850.524770,11202.834434,24773.437500,0.000867,2.821577e-06,307.423673,38.223540,0.000000


In [49]:
# ===== SAVE RAW FEATURES =====
merged_df.to_csv("test_features_raw.csv", index=False)

print("Test features saved as test_features_raw.csv")

Test features saved as test_features_raw.csv


In [50]:
FEATURE_COLUMNS = [
    "EDA_SCR_count",
    "EDA_Phasic_AUC",
    "EDA_Tonic_Mean",
    "EDA_Tonic_Diff",
    "HRV_RMSSD",
    "HRV_SDNN",
    "HRV_MeanNN",
    "HRV_LF",
    "HRV_HF",
    "HRV_LFHF",
    "HRV_RMSSD_Diff"
]

X = merged_df[FEATURE_COLUMNS].copy()

In [51]:
import joblib

# Load the scaler
scaler = joblib.load("/content/drive/MyDrive/major/subject_scaler.pkl")

# Ensure X has the exact columns and order the scaler was trained on
if hasattr(scaler, 'feature_names_in_'):
    X = X[scaler.feature_names_in_]

X_scaled = scaler.transform(X)
print("Features scaled successfully.")

Features scaled successfully.


In [52]:
model = joblib.load("/content/drive/MyDrive/major/lda_model.pkl")

In [53]:
# Drop rows with NaNs before prediction to avoid ValueError
valid_mask = ~np.isnan(X_scaled).any(axis=1)
X_scaled_clean = X_scaled[valid_mask]

# Filter merged_df to match the rows we are predicting
merged_df = merged_df[valid_mask].copy()

# Perform prediction
predictions = model.predict(X_scaled_clean)

# If model outputs probabilities, convert to binary labels
if predictions.ndim > 1 or (predictions.max() <= 1 and predictions.min() >= 0):
    pred_labels = (predictions > 0.5).astype(int)
else:
    pred_labels = predictions

merged_df["Predicted_Label"] = pred_labels
print(f"Predictions completed for {len(merged_df)} samples.")

Predictions completed for 103 samples.


In [54]:
merged_df.to_csv("final_predictions.csv", index=False)

print("Predictions saved as final_predictions.csv")

Predictions saved as final_predictions.csv


In [55]:
print("=== Prediction Results ===")
# Display the count of each predicted class
print(merged_df['Predicted_Label'].value_counts())

# Display the first few rows of the final results
display(merged_df[['Time', 'EDA_Tonic_Mean', 'HRV_RMSSD', 'Predicted_Label']].head(10))

=== Prediction Results ===
Predicted_Label
0    103
Name: count, dtype: int64


,Time,EDA_Tonic_Mean,HRV_RMSSD,Predicted_Label
0,0.0,2082.367377,6705.786913,0
1,60.0,2101.820558,6705.786913,0
2,120.0,2122.767895,12097.907877,0
3,180.0,2144.672340,14850.524770,0
4,240.0,2182.895880,14850.524770,0
5,300.0,2213.104616,14850.524770,0
6,360.0,2226.204529,14850.524770,0
7,420.0,2231.753382,14850.524770,0
8,480.0,2165.045089,14850.524770,0
9,540.0,2083.927579,14850.524770,0


### Label Mapping
We will map the binary `Predicted_Label` to descriptive strings: `0` for **Non-Stress** and `1` for **Stress**.

In [58]:
label_map = {0: 'Non-Stress', 1: 'Stress'}
merged_df['Stress_Status'] = merged_df['Predicted_Label'].map(label_map)

# Display summary
print("Current Status Summary:")
print(merged_df['Stress_Status'].value_counts())
display(merged_df[['Time', 'Stress_Status']].tail())


Current Status Summary:
Stress_Status
Non-Stress    103
Name: count, dtype: int64


,Time,Stress_Status
98,5880.0,Non-Stress
99,5940.0,Non-Stress
100,6000.0,Non-Stress
101,6060.0,Non-Stress
102,6120.0,Non-Stress


### Upload to Firebase
To send the output to Firebase, you need to:
1. Go to your Firebase Console > Project Settings > Service Accounts.
2. Click **Generate new private key** and download the JSON file.
3. Upload that file to the Colab file browser (left panel) and rename it to `firebase_key.json`.

In [59]:
!pip install firebase-admin

In [62]:
import firebase_admin
from firebase_admin import credentials, db

# Path to your uploaded service account key
FIREBASE_KEY_PATH = '/content/drive/MyDrive/major/firebase_key.json'
# Your Realtime Database URL (e.g., https://your-project-id.firebaseio.com/)
DATABASE_URL = 'https://mindcare-97d3b-default-rtdb.asia-southeast1.firebasedatabase.app/'

try:
    if not firebase_admin._apps:
        cred = credentials.Certificate(FIREBASE_KEY_PATH)
        firebase_admin.initialize_app(cred, {
            'databaseURL': DATABASE_URL
        })

    # Convert the latest prediction to a dictionary for upload
    # We'll upload the most recent window result
    latest_data = merged_df.iloc[-1].to_dict()

    ref = db.reference('stress_monitoring/latest')
    ref.set(latest_data)

    print("Successfully uploaded latest prediction to Firebase!")
except Exception as e:
    print(f"Firebase Error: {e}")
    print("Make sure you uploaded 'firebase_key.json' and set your DATABASE_URL correctly.")

Successfully uploaded latest prediction to Firebase!
